# #1 Import libraries

In [78]:
import pandas as pd
import numpy as np
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

# #2 Import dataset

In [94]:
df_raw = pd.read_csv("public_events_toronto.csv")
df_raw.head()


,Event Name,Short Name,Event Description,Event Category,Event Start Date,Event Time Starts,Event End Date,Event Time Ends,Free Event,Accesible Event,Event Website,Event Email,Event Telephone,Location,Longitude,Latitude,Event Attendance,Mean Attendance per Day
0,St. Patrick's Day Parade,St. Patrick's Day,Festive parade returning to the streets of dow...,Parade / Culture,2022-03-20,12:00:00,2022-03-20,3:00:00 PM,Yes,Yes,stpatrickstoronto.com,info@stpatrickstoronto.com,+1 416-996-8431,St. George St & Bloor St W,43.667123,-79.399548,300000,300000
1,Toronto Blue Jays Opening Day,Blue Jays Day 1,MLB season home opener against Texas Rangers a...,Sports,2022-04-08,19:00:00,2022-04-08,10:30:00 PM,No,Yes,mlb.com/bluejays,fanfeedback@bluejays.com,+1 416-341-1000,Rogers Centre,43.641660,-79.389198,45022,45022
2,National Home Show,Home Show Day 1,Consumer show for home improvement and decor.,Exhibition,2022-04-15,10:00:00,2022-04-15,8:00:00 PM,No,Yes,nationalhomeshow.com,info@homeshow.ca,+1 416-644-5400,Enercare Centre,43.635356,-79.413237,150000,15000
3,National Home Show,Home Show Day 2,Consumer show for home improvement and decor.,Exhibition,2022-04-16,10:00:00,2022-04-16,8:00:00 PM,No,Yes,nationalhomeshow.com,info@homeshow.ca,+1 416-644-5400,Enercare Centre,43.635356,-79.413237,150000,15000
4,National Home Show,Home Show Day 3,Consumer show for home improvement and decor.,Exhibition,2022-04-17,10:00:00,2022-04-17,6:00:00 PM,No,Yes,nationalhomeshow.com,info@homeshow.ca,+1 416-644-5400,Enercare Centre,43.635356,-79.413237,150000,15000


# #3 Data info & Data preprocessing



3.1 data info

In [95]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Event Name               350 non-null    object 
 1   Short Name               350 non-null    object 
 2   Event Description        350 non-null    object 
 3   Event Category           350 non-null    object 
 4   Event Start Date         350 non-null    object 
 5   Event Time Starts        350 non-null    object 
 6   Event End Date           350 non-null    object 
 7   Event Time Ends          350 non-null    object 
 8   Free Event               350 non-null    object 
 9   Accesible Event          350 non-null    object 
 10  Event Website            350 non-null    object 
 11  Event Email              346 non-null    object 
 12  Event Telephone          350 non-null    object 
 13  Location                 350 non-null    object 
 14  Longitude                3

In [96]:
df = df_raw.copy()
df['Event Category'].unique()

array(['Parade / Culture', 'Sports', 'Exhibition', 'Sports / Outdoor',
       'Culture / History', 'Community', 'Sports / Charity',
       'Arts / Culture', 'Arts / Music', 'Community / Music',
       'Community / Festival', 'Festival / Dance', 'Sports / Racing',
       'Festival / Culture', 'Arts / Community', 'Music Festival',
       'Food / Music', 'Food / Festival', 'Community / Car Show',
       'Concert', 'Exhibition / Fair', 'Convention', 'Arts / Cinema',
       'Community / Food', 'Sports / Fitness', 'Agricultural / Fair',
       'Parade / Holiday', 'Community / Parade', 'Holiday / Festival',
       'Parade', 'History', 'Holiday', 'Fair', 'Arts', 'Arts / Theater',
       'Arts / Literacy', 'Festival', 'Convention / Tech',
       'Special Event'], dtype=object)

In [97]:
print("START TIMES", df['Event Time Starts'].unique())
print("\nEND TIMES:" , df['Event Time Ends'].unique())

START TIMES ['12:00:00' '19:00:00' '10:00:00' '9:00:00 AM' '7:00:00 AM' '12:00:00 PM'
 '6:00:00 AM' '10:00:00 AM' '8:00:00' '13:00:00' '8:00:00 AM' '5:00:00 PM'
 '6:00:00 PM' '11:00:00 AM' '4:00:00 PM' '20:00:00' '19:30:00' '12:30:00'
 '10:30:00 AM' '7:00:00 PM' '7:30:00 AM' '1:00:00 PM' '12:30:00 PM'
 '3:00:00 PM' '7:30:00 PM' '9:00:00 PM' '11:00:00' '18:00:00' '15:00:00'
 '10:30:00' '19:07:00' '14:00:00' '9:00:00' '7:30:00' '16:00:00'
 '17:00:00' '7:00:00' '6:00:00']

END TIMES: ['3:00:00 PM' '10:30:00 PM' '8:00:00 PM' '6:00:00 PM' '2:00:00 PM'
 '7:00:00 PM' '5:00:00 PM' '1:00:00 PM' '11:00:00 PM' '11:59:00 PM'
 '10:00:00 PM' '9:00:00 PM' '21:00:00' '19:00:00' '17:00:00' '3:30:00 PM'
 '11:30:00 PM' '7:00:00 AM' '4:00:00 PM' '12:30:00 AM' '9:30:00 PM'
 '1:00:00 AM' '4:00:00 AM' '4:30:00 PM' '20:00:00']


3.2 data normalization

In [98]:
# Normaliztion of column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [99]:
# Normalization of event categories
df["event_category"] = df["event_category"].str.split("/").str[0].str.strip()
df['event_category'].unique()

array(['Parade', 'Sports', 'Exhibition', 'Culture', 'Community', 'Arts',
       'Festival', 'Music Festival', 'Food', 'Concert', 'Convention',
       'Agricultural', 'Holiday', 'History', 'Fair', 'Special Event'],
      dtype=object)

In [100]:
# Transformation of timme
df['event_start_date'] = pd.to_datetime(df['event_start_date'], errors='coerce')
df['event_end_date'] = pd.to_datetime(df['event_end_date'], errors='coerce')

# Normalization of time
df["event_time_starts"] = pd.to_datetime(df["event_time_starts"], errors="coerce").dt.round("H").dt.strftime("%H:00")
df["event_time_ends"] = pd.to_datetime(df["event_time_ends"], errors="coerce").dt.round("H").dt.strftime("%H:00")



/tmp/ipython-input-487086148.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["event_time_starts"] = pd.to_datetime(df["event_time_starts"], errors="coerce").dt.round("H").dt.strftime("%H:00")
/tmp/ipython-input-487086148.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["event_time_starts"] = pd.to_datetime(df["event_time_starts"], errors="coerce").dt.round("H").dt.strftime("%H:00")
/tmp/ipython-input-487086148.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["event_time_ends"] = pd.to_datetime(df["event_time_ends"], errors="coerce").dt.round("H").dt.strftime("%H:00")
/tmp/ipython-input-487086148.py:7: FutureWarning: 'H' is deprecated and w

In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   event_name               350 non-null    object        
 1   short_name               350 non-null    object        
 2   event_description        350 non-null    object        
 3   event_category           350 non-null    object        
 4   event_start_date         350 non-null    datetime64[ns]
 5   event_time_starts        350 non-null    object        
 6   event_end_date           350 non-null    datetime64[ns]
 7   event_time_ends          350 non-null    object        
 8   free_event               350 non-null    object        
 9   accesible_event          350 non-null    object        
 10  event_website            350 non-null    object        
 11  event_email              346 non-null    object        
 12  event_telephone          350 non-nul

3.3 data cleaning

In [102]:
df = df.drop_duplicates()

In [103]:
# Filtering by interested dates (OCT-2022 to SEP-2024)

start_range = pd.Timestamp("2022-10-01")
end_range = pd.Timestamp("2024-09-30")

df = df[
    (df["event_start_date"] >= start_range) &
    (df["event_start_date"] <= end_range)
]

3.4 Feature extraction

In [104]:
df["event_id"] = df.index.astype(str)
df = df[["event_id"] + [col for col in df.columns if col != "event_id"]]

# Creation of new columns (YEAR, MONTH and DAY) based on event dates
df["event_start_date_y"] = df["event_start_date"].dt.year
df["event_start_date_m"] = df["event_start_date"].dt.month
df["event_start_date_d"] = df["event_start_date"].dt.day

df["event_end_date_y"] = df["event_end_date"].dt.year
df["event_end_date_m"] = df["event_end_date"].dt.month
df["event_end_date_d"] = df["event_end_date"].dt.day

In [105]:
df.head(10)

,event_id,event_name,short_name,event_description,event_category,event_start_date,event_time_starts,event_end_date,event_time_ends,free_event,...,longitude,latitude,event_attendance,mean_attendance_per_day,event_start_date_y,event_start_date_m,event_start_date_d,event_end_date_y,event_end_date_m,event_end_date_d
97,97,Nuit Blanche Toronto,Nuit Blanche Day 1,City-wide all-night contemporary art celebration.,Arts,2022-10-01,19:00,2022-10-02,07:00,Yes,...,43.813164,-79.247122,1200000,1200000,2022,10,1,2022,10,2
98,98,Toronto Maple Leafs Opener,Maple Leafs Day 1,NHL regular season home opener at Scotiabank A...,Sports,2022-10-13,20:00,2022-10-13,22:00,No,...,43.643434,-79.379078,19000,19000,2022,10,13,2022,10,13
99,99,Toronto Waterfront Marathon,Marathon Day 1,Elite international marathon through downtown ...,Sports,2022-10-16,08:00,2022-10-16,15:00,No,...,43.656322,-79.380916,25000,25000,2022,10,16,2022,10,16
100,100,Royal Agricultural Winter Fair,The Royal Day 1,The world's largest combined indoor agricultur...,Agricultural,2022-11-04,09:00,2022-11-04,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,4,2022,11,4
101,101,Royal Agricultural Winter Fair,The Royal Day 2,The world's largest combined indoor agricultur...,Agricultural,2022-11-05,09:00,2022-11-05,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,5,2022,11,5
102,102,Royal Agricultural Winter Fair,The Royal Day 3,The world's largest combined indoor agricultur...,Agricultural,2022-11-06,09:00,2022-11-06,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,6,2022,11,6
103,103,Royal Agricultural Winter Fair,The Royal Day 4,The world's largest combined indoor agricultur...,Agricultural,2022-11-07,09:00,2022-11-07,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,7,2022,11,7
104,104,Royal Agricultural Winter Fair,The Royal Day 5,The world's largest combined indoor agricultur...,Agricultural,2022-11-08,09:00,2022-11-08,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,8,2022,11,8
105,105,Royal Agricultural Winter Fair,The Royal Day 6,The world's largest combined indoor agricultur...,Agricultural,2022-11-09,09:00,2022-11-09,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,9,2022,11,9
106,106,Royal Agricultural Winter Fair,The Royal Day 7,The world's largest combined indoor agricultur...,Agricultural,2022-11-10,09:00,2022-11-10,21:00,No,...,43.633521,-79.417724,300000,30000,2022,11,10,2022,11,10


# #4. Data written in dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/public_events

In [106]:
df.to_csv("public_events_cleaned.csv", index=False)

In [ ]:
output_path = "/dbfs/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/public_events_cleaned"

df.to_parquet(output_path, index=False)